# 10장 · 미분의 응용 — 역전파 직관 🎓 (마지막)

웹 챕터 **"역전파 직관"** 과 짝을 이루는 Python 실습입니다.
계산그래프 L=(x·y+z)² 의 순전파·역전파를 직접 코딩하고, SymPy 정답과 검산합니다.
(미니 autograd, 딥러닝 프레임워크가 하는 일의 축소판!)

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch10')

## 1. 순전파: 값 계산 (왼→오)

L = (x·y + z)² 를 세 노드로 쪼갭니다: p=x·y, q=p+z, L=q².

In [ ]:
x, y, z = 2.0, 3.0, 1.0

p = x * y        # 곱셈 노드
q = p + z        # 덧셈 노드
L = q ** 2       # 제곱 노드
print(f'p = {p},  q = {q},  L = {L}')

## 2. 역전파: 기울기 전파 (오→왼)

출력 기울기 1 에서 시작해 '상류 기울기 × 국소 미분'을 뒤로 이어붙입니다.
- 제곱: 국소미분 2q
- 덧셈: 그대로 복사
- 곱셈: 상대편 값 곱하기

각 규칙엔 이유가 있어요: **덧셈**은 국소미분이 1이라 기울기가 그대로 통과(복사), **곱셈**은 ∂(x·y)/∂x = y 라 상대편 값이 곱해집니다(8장 편미분). 이 "상류 기울기 × 국소미분" 이어붙이기가 곧 **연쇄법칙**이에요.

In [ ]:
gL = 1.0            # ∂L/∂L
gq = gL * (2 * q)  # 제곱 노드: 국소미분 2q
gp = gq * 1        # 덧셈: 복사
gz = gq * 1        # 덧셈: 복사
gx = gp * y        # 곱셈: 상대편 y
gy = gp * x        # 곱셈: 상대편 x

print(f'∂L/∂q = {gq}')
print(f'∂L/∂p = {gp},  ∂L/∂z = {gz}')
print(f'∂L/∂x = {gx},  ∂L/∂y = {gy}')

## 3. SymPy 로 검산

손으로 역전파한 기울기가 진짜 편미분과 같은지 확인합니다.

In [ ]:
import sympy as sp

sx, sy, sz = sp.symbols('x y z')
Lsym = (sx * sy + sz) ** 2
at = {sx: 2, sy: 3, sz: 1}

print('∂L/∂x :', sp.diff(Lsym, sx).subs(at), ' (역전파:', gx, ')')
print('∂L/∂y :', sp.diff(Lsym, sy).subs(at), ' (역전파:', gy, ')')
print('∂L/∂z :', sp.diff(Lsym, sz).subs(at), ' (역전파:', gz, ')')
ok = (sp.diff(Lsym, sx).subs(at) == gx and sp.diff(Lsym, sy).subs(at) == gy and sp.diff(Lsym, sz).subs(at) == gz)
print('모두 일치?', ok)

## 4. 미니 학습 루프: 역전파 + 경사하강

이 계산그래프의 L 을 손실이라 보고, x·y·z 를 가중치처럼 경사하강으로 줄여봅니다.
역전파(기울기) → 갱신(w ← w − η·기울기) 을 반복하면 L 이 0 으로 내려갑니다.

In [ ]:
x, y, z = 2.0, 3.0, 1.0
eta = 0.02
for i in range(20):
    # 순전파
    p = x * y; q = p + z; L = q**2
    # 역전파
    gq = 2*q; gx = gq*y; gy = gq*x; gz = gq
    # 갱신 (경사하강)
    x -= eta*gx; y -= eta*gy; z -= eta*gz
    if i % 4 == 0:
        print(f'{i:2d}걸음: L = {L:.4f}')
print(f'최종 L = {(x*y+z)**2:.6f}  (0 에 수렴)')

## 5. 직접 풀어보기 ✏️

계산그래프 p=x·y, q=p+z, L=q² 에서 **x=1, y=4, z=2 일 때 ∂L/∂x** 를 구해 보세요.
(웹 문제와 같습니다. 답은 48!)

In [ ]:
# TODO: q=x·y+z, ∂L/∂q=2q, ∂L/∂x = ∂L/∂q · y 로 구해 answer 에 담으세요.

x, y, z = 1.0, 4.0, 2.0
q = x*y + z
answer = (2*q) * y      # ∂L/∂x = 2q · y
print('정답:', answer)
assert abs(answer - 48) < 1e-9, '다시! q=6, ∂L/∂q=12, ∂L/∂x=12·4=48.'
print('✅ 잘했어요! q=6 → ∂L/∂q=12 → ∂L/∂x=12·4=48. (곱셈 노드는 상대편 y 곱)')

---
## 🎓 완주 축하합니다!

접선의 기울기에서 시작해 **극한 → 도함수 → 미분 공식 → 연쇄법칙 → 편미분 → 그래디언트 →
경사하강 → 역전파** 까지 왔습니다. 이제 'AI가 어떻게 배우는가'를 미분으로 설명할 수 있어요.

이 미니 autograd 가 PyTorch·TensorFlow 가 내부에서 하는 일의 축소판입니다. 수고했어요! 🙌